In [1]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

INPUT = Path("MLB Game Get-In Prices.csv")
OUTDIR = Path("damilareshonuyi/Desktop/MLB Road Team Impact/data")


In [2]:
def clean_price(x):
    if pd.isna(x):
        return np.nan
    return float(re.sub(r"[^0-9.]", "", str(x)))


def extract_matchup(event):
    if not isinstance(event, str) or " at " not in event:
        return pd.Series({"road_team": np.nan, "home_team_event": np.nan})
    road, home = event.split(" at ", 1)
    return pd.Series({"road_team": road.strip(), "home_team_event": home.strip()})


In [3]:
def main():
    df = pd.read_csv(INPUT)
    df["price"] = df["Get-in Price"].apply(clean_price)
    df["date"] = pd.to_datetime(df["Date"], errors="coerce")

    teams = df["Event"].apply(extract_matchup)
    df = pd.concat([df, teams], axis=1)

    # Exclude non-standard rows such as All-Star Game and rows without a road/home matchup.
    df = df.dropna(subset=["road_team", "home_team_event", "price", "date"]).copy()
    df = df[df["price"] > 0].copy()

    # Use Performer as home team when available; fallback to parsed home team.
    df["home_team"] = df["Performer"].where(
        df["Performer"].notna(), df["home_team_event"]
    )
    df["month"] = df["date"].dt.month_name()
    df["day_of_week"] = df["date"].dt.day_name()
    df["weekend"] = df["date"].dt.dayofweek.isin([5, 6]).astype(int)
    df["log_price"] = np.log(df["price"])

    # Require each road team to appear at least twice to avoid one-game-only estimates dominating.
    counts = df["road_team"].value_counts()
    eligible_road_teams = counts[counts >= 2].index
    model_df = df[df["road_team"].isin(eligible_road_teams)].copy()

    formula = (
        "log_price ~ C(road_team) + C(home_team) + C(month) + C(day_of_week) + weekend"
    )
    model = smf.ols(formula=formula, data=model_df).fit(cov_type="HC3")
    OUTDIR.mkdir(parents=True, exist_ok=True)
    # Recover road-team fixed effects. Statsmodels omits one base category; its effect is 0.
    all_teams = sorted(model_df["road_team"].unique())
    effects = {team: 0.0 for team in all_teams}
    pvals = {team: np.nan for team in all_teams}
    ses = {team: np.nan for team in all_teams}
    for name, coef in model.params.items():
        prefix = "C(road_team)[T."
        if name.startswith(prefix):
            team = name[len(prefix) : -1]
            effects[team] = coef
            pvals[team] = model.pvalues[name]
            ses[team] = model.bse[name]
    # Center effects around the average road-team effect so results are relative to an average road team,
    # not relative to the alphabetically omitted category.
    avg_effect = np.mean(list(effects.values()))
    median_price = model_df["price"].median()

    rows = []
    raw_avg = model_df.groupby("road_team")["price"].mean()
    n_games = model_df.groupby("road_team").size()
    for team in all_teams:
        centered_log_effect = effects[team] - avg_effect
        pct_premium = np.expm1(centered_log_effect) * 100
        dollar_premium = median_price * np.expm1(centered_log_effect)
        rows.append(
            {
                "road_team": team,
                "games": int(n_games[team]),
                "raw_avg_get_in_price": raw_avg[team],
                "log_effect_centered": centered_log_effect,
                "estimated_pct_premium_vs_avg_road_team": pct_premium,
                "estimated_dollar_premium_vs_avg_road_team": dollar_premium,
                "model_p_value_vs_base_category": pvals[team],
                "robust_standard_error_vs_base_category": ses[team],
            }
        )

    effects_df = pd.DataFrame(rows).sort_values(
        "estimated_dollar_premium_vs_avg_road_team", ascending=False
    )
    effects_df.to_csv(OUTDIR / "road_team_effects.csv", index=False)

    with open(OUTDIR / "regression_summary.txt", "w") as f:
        f.write(model.summary().as_text())
        f.write("\n\nNotes:\n")
        f.write(f"Rows used in model: {len(model_df)}\n")
        f.write(f"Road teams modeled: {model_df['road_team'].nunique()}\n")
        f.write(
            f"Median get-in price used for dollar premium conversion: ${median_price:.2f}\n"
        )
        f.write(
            "Dollar premiums are relative to the average road team after controls for home team, month, day of week, and weekend.\n"
        )

    # Visual 1: raw averages
    top_raw = raw_avg.sort_values(ascending=False).head(15).sort_values()
    plt.figure(figsize=(10, 7))
    plt.barh(top_raw.index, top_raw.values)
    plt.xlabel("Average get-in price ($)")
    plt.title("Raw average get-in price by road team, top 15")
    plt.tight_layout()
    plt.savefig(OUTDIR / "raw_average_price_by_road_team.png", dpi=200)
    plt.close()

    # Visual 2: model-estimated premiums
    top_effects = effects_df.head(15).sort_values(
        "estimated_dollar_premium_vs_avg_road_team"
    )
    plt.figure(figsize=(10, 7))
    plt.barh(
        top_effects["road_team"],
        top_effects["estimated_dollar_premium_vs_avg_road_team"],
    )
    plt.xlabel("Estimated get-in price premium vs average road team ($)")
    plt.title("Estimated road-team ticket premium after controls, top 15")
    plt.tight_layout()
    plt.savefig(OUTDIR / "estimated_road_team_premium.png", dpi=200)
    plt.close()

    # Visual 3: actual vs fitted
    fitted = np.exp(model.fittedvalues)
    actual = model_df["price"]
    plt.figure(figsize=(8, 6))
    plt.scatter(fitted, actual, alpha=0.55)
    maxv = max(float(fitted.max()), float(actual.max()))
    plt.plot([0, maxv], [0, maxv], linestyle="--")
    plt.xlabel("Fitted get-in price ($)")
    plt.ylabel("Actual get-in price ($)")
    plt.title("Model fit: actual vs fitted prices")
    plt.tight_layout()
    plt.savefig(OUTDIR / "model_actual_vs_fitted.png", dpi=200)
    plt.close()

    print("Regression complete.")
    print(f"Rows used: {len(model_df)}")
    print(f"R-squared: {model.rsquared:.3f}")
    print("\nTop estimated road-team premiums:")
    print(
        effects_df[
            [
                "road_team",
                "games",
                "raw_avg_get_in_price",
                "estimated_dollar_premium_vs_avg_road_team",
                "estimated_pct_premium_vs_avg_road_team",
            ]
        ]
        .head(10)
        .to_string(index=False)
    )


In [4]:
if __name__ == "__main__":
    main()


/opt/base-uv/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 69, but rank is 68
  warnings.warn('covariance of constraints does not have full '


Regression complete.
Rows used: 1622
R-squared: 0.772

Top estimated road-team premiums:
            road_team  games  raw_avg_get_in_price  estimated_dollar_premium_vs_avg_road_team  estimated_pct_premium_vs_avg_road_team
  Los Angeles Dodgers     53             63.716981                                  26.965690                               99.872926
     New York Yankees     53             62.867925                                  21.218645                               78.587575
         Chicago Cubs     56             39.428571                                  10.234074                               37.903978
       Boston Red Sox     54             41.018519                                   7.106964                               26.322088
        New York Mets     52             38.903846                                   5.645485                               20.909204
Philadelphia Phillies     56             30.821429                                   3.246101              